# 1 month salt budget test for panant-01 control, shelf only

Converting salt budget terms into freshwater fluxes so I can compare the magnitude of these terms to the freshwater volume fluxes.

In [1]:
import glob
import dask.distributed as dsk
import matplotlib.pyplot as plt
import xarray as xr
import cf_xarray as cfxr
import numpy as np
import xesmf as xe
from matplotlib.colors import to_rgba

import cmocean as cm
import cartopy.crs as ccrs

import cartopy.feature as cft
import matplotlib.path as mpath

import warnings # ignore these warnings
warnings.filterwarnings("ignore", category = FutureWarning)
warnings.filterwarnings("ignore", category = UserWarning)
warnings.filterwarnings("ignore", category = RuntimeWarning)

In [2]:
lat_slice = slice(-79,-59)

# Importing data
paths_1_C = np.sort(glob.glob('/scratch/x77/kc5856/mom6/archive/panant-01-zstar-ACCESSyr2-test-pan01_test_01-f7e0ea96/output091/*.ocean_month.nc'))
paths_2_C = np.sort(glob.glob('/scratch/x77/kc5856/mom6/archive/panant-01-zstar-ACCESSyr2-test-pan01_test_01-f7e0ea96/output091/*.ocean_month_z.nc'))

paths_1_M = np.sort(glob.glob('/scratch/x77/kc5856/mom6/archive/panant-ssp126-sofia-expt-3393fa55/output091/*.ocean_month.nc'))
paths_2_M = np.sort(glob.glob('/scratch/x77/kc5856/mom6/archive/panant-ssp126-sofia-expt-3393fa55/output091/*.ocean_month_z.nc'))

In [3]:
# Time variant variables
def preprocess_1(ds):
    ds = ds[['wfo','lrunoff']].sel(yh=lat_slice)
    return ds

def preprocess_2(ds):
    ds = ds[['so','S_advection_xy','Sh_tendency_vert_remap','osalttend','osaltpmdiff','osaltdiff','boundary_forcing_salt_tendency']].sel(yh=lat_slice)
    return ds

data_1_C = xr.open_mfdataset(paths_1_C, preprocess = preprocess_1, chunks = 'auto')
data_2_C = xr.open_mfdataset(paths_2_C, preprocess = preprocess_2, chunks = 'auto')

data_1_M = xr.open_mfdataset(paths_1_M, preprocess = preprocess_1, chunks = 'auto')
data_2_M = xr.open_mfdataset(paths_2_M, preprocess = preprocess_2, chunks = 'auto')

In [4]:
# Shelf mask

def shelf_mask_isobath(var, model_dict):

    paths = {
             "mom5": "/g/data/ik11/grids/Antarctic_slope_contour_1000m.npz",
             "mom6_01": "/g/data/ik11/grids/Antarctic_slope_contour_1000m_MOM6_01deg.nc",
            "mom6_005": "/g/data/ik11/grids/Antarctic_slope_contour_1000m_MOM6_005deg.nc"
             }

    var = var.cf.sel({'latitude': slice(-90, -59)})

    if paths[model_dict][-3:] == '.nc':
        shelf_mask = xr.open_dataset(paths[model_dict])['contour_masked_above']
    else:
        contour_file = np.load(paths[model_dict])
        shelf_mask = xr.DataArray(contour_file['contour_masked_above'],
                                  coords = var.coords, 
                                  dims = var.dims,
                                  name = 'contour_masked_above')
    
    shelf_mask = xr.where(shelf_mask == 0, 1, 0)
    masked_var = var * shelf_mask
    
    return masked_var, shelf_mask

In [5]:
depth = xr.open_dataset('/g/data/ol01/outputs/mom6-panan/panant-01-zstar-ACCESSyr2/output050/20050501.ocean_static.nc')['deptho']
land_mask = (0 * depth).fillna(1)
land = xr.where(np.isnan(depth.rename('land_1')), 1, np.nan)
depth_shelf, shelf_mask = shelf_mask_isobath(depth, 'mom6_01')

In [6]:
# testing individual diagnostics
tendency_C = data_2_C.osalttend
adv_horizontal_C = data_2_C.S_advection_xy
adv_vertical_C = data_2_C.Sh_tendency_vert_remap
surface_forcing_C = data_2_C.boundary_forcing_salt_tendency
diff_1_C = data_2_C.osaltdiff
diff_2_C = data_2_C.osaltpmdiff

# testing individual diagnostics
tendency_M = data_2_M.osalttend
adv_horizontal_M = data_2_M.S_advection_xy
adv_vertical_M = data_2_M.Sh_tendency_vert_remap
surface_forcing_M = data_2_M.boundary_forcing_salt_tendency
diff_1_M = data_2_M.osaltdiff
diff_2_M = data_2_M.osaltpmdiff

In [7]:
# Getting SSS
SSS_C = data_2_C.so.isel(time=0,z_l=0)
SSS_M = data_2_M.so.isel(time=0,z_l=0)

In [8]:
# Converting diagnostics to freshwater fluxes
tendency_FWF_C = -tendency_C*1000/SSS_C # negative sign makes this a salt -> FW flux conversion
adv_horizontal_FWF_C = -adv_horizontal_C*1000/SSS_C
adv_vertical_FWF_C = -adv_vertical_C*1000/SSS_C
surface_forcing_FWF_C = -surface_forcing_C*1000/SSS_C
diff_1_FWF_C = -diff_1_C*1000/SSS_C
diff_2_FWF_C = -diff_2_C*1000/SSS_C

In [9]:
# Converting diagnostics to freshwater fluxes
tendency_FWF_M = -tendency_M*1000/SSS_M  # negative sign makes this a salt -> FW flux conversion
adv_horizontal_FWF_M = -adv_horizontal_M*1000/SSS_M
adv_vertical_FWF_M = -adv_vertical_M*1000/SSS_M
surface_forcing_FWF_M = -surface_forcing_M*1000/SSS_M
diff_1_FWF_M = -diff_1_M*1000/SSS_M
diff_2_FWF_M = -diff_2_M*1000/SSS_M

In [10]:
zdim='z_l'

In [11]:
tendency_plot_C = tendency_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
adv_hor_plot_C = adv_horizontal_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
adv_vert_plot_C = adv_vertical_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
surface_plot_C = surface_forcing_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
diff_1_plot_C = diff_1_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
diff_2_plot_C = diff_2_FWF_C.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()

tendency_plot_M = tendency_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
adv_hor_plot_M = adv_horizontal_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
adv_vert_plot_M = adv_vertical_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
surface_plot_M = surface_forcing_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
diff_1_plot_M = diff_1_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()
diff_2_plot_M = diff_2_FWF_M.isel(time=0).where(shelf_mask == 1).sum(zdim).squeeze()

In [12]:
tendency_plot_C = tendency_plot_C.compute()
adv_hor_plot_C = adv_hor_plot_C.compute()
adv_vert_plot_C = adv_vert_plot_C.compute()
surface_plot_C = surface_plot_C.compute()
diff_1_plot_C = diff_1_plot_C.compute()
diff_2_plot_C = diff_2_plot_C.compute()

In [13]:
tendency_plot_M = tendency_plot_M.compute()
adv_hor_plot_M = adv_hor_plot_M.compute()
adv_vert_plot_M = adv_vert_plot_M.compute()
surface_plot_M = surface_plot_M.compute()
diff_1_plot_M = diff_1_plot_M.compute()
diff_2_plot_M = diff_2_plot_M.compute()

In [14]:
# Doing spatial average
area = xr.open_dataset('/g/data/ol01/outputs/mom6-panan/panant-01-zstar-ACCESSyr2/output050/20050501.ocean_static.nc')['areacello']
var = data_2.so.isel(time=0,z_l=0)
area_corr = area*(var*0 +1)
area_corr.plot()

NameError: name 'data_2' is not defined

In [ ]:
shelf_area = area_corr.where(shelf_mask == 1).sum(['xh', 'yh'])

tendency_ave_C = (area_corr * tendency_plot_C).sum(['xh', 'yh']) / shelf_area
adv_hor_ave_C = (area_corr * adv_hor_plot_C).sum(['xh', 'yh']) / shelf_area
adv_vert_ave_C = (area_corr * adv_vert_plot_C).sum(['xh', 'yh']) / shelf_area
surface_ave_C = (area_corr * surface_plot_C).sum(['xh', 'yh']) / shelf_area
diff_1_ave_C = (area_corr * diff_1_plot_C).sum(['xh', 'yh']) / shelf_area
diff_2_ave_C = (area_corr * diff_2_plot_C).sum(['xh', 'yh']) / shelf_area

tendency_ave_M = (area_corr * tendency_plot_M).sum(['xh', 'yh']) / shelf_area
adv_hor_ave_M = (area_corr * adv_hor_plot_M).sum(['xh', 'yh']) / shelf_area
adv_vert_ave_M = (area_corr * adv_vert_plot_M).sum(['xh', 'yh']) / shelf_area
surface_ave_M = (area_corr * surface_plot_M).sum(['xh', 'yh']) / shelf_area
diff_1_ave_M = (area_corr * diff_1_plot_M).sum(['xh', 'yh']) / shelf_area
diff_2_ave_M = (area_corr * diff_2_plot_M).sum(['xh', 'yh']) / shelf_area

In [ ]:
tendency_ave_C = tendency_ave_C.load()
adv_hor_ave_C = adv_hor_ave_C.load()
adv_vert_ave_C = adv_vert_ave_C.load()
surface_ave_C = surface_ave_C.load()
diff_1_ave_C = diff_1_ave_C.load()
diff_2_ave_C = diff_2_ave_C.load()

In [ ]:
tendency_ave_M = tendency_ave_M.load()
adv_hor_ave_M = adv_hor_ave_M.load()
adv_vert_ave_M = adv_vert_ave_M.load()
surface_ave_M = surface_ave_M.load()
diff_1_ave_M = diff_1_ave_M.load()
diff_2_ave_M = diff_2_ave_M.load()

**On average, how much salt per square metre is being added/removed from the shelf (full-depth integrated), due to...**

In [ ]:
print("**Control**","\ntendency =", tendency_ave_C.values, "\nhorizontal advection =", adv_hor_ave_C.values, "\nvertical advection =", adv_vert_ave_C.values, "\nsurface forcing =", 
      surface_ave_C.values, "\ndiffusion 1 =", diff_1_ave_C.values, "\ndiffusion 2 =", diff_2_ave_C.values)

print("**Melt**",
      "\ntendency =", tendency_ave_M.values,
      "\nhorizontal advection =", adv_hor_ave_M.values,
      "\nvertical advection =", adv_vert_ave_M.values,
      "\nsurface forcing =", surface_ave_M.values,
      "\ndiffusion 1 =", diff_1_ave_M.values,
      "\ndiffusion 2 =", diff_2_ave_M.values)

In [ ]:
# Checking closure
print("**Control**")
print("LHS =", tendency_ave_C.values,
      "\nRHS =", (adv_hor_ave_C + adv_vert_ave_C + surface_ave_C + diff_1_ave_C + diff_2_ave_C).values)

print("\n**Melt**")
print("LHS =", tendency_ave_M.values,
      "\nRHS =", (adv_hor_ave_M + adv_vert_ave_M + surface_ave_M + diff_1_ave_M + diff_2_ave_M).values)

So the change due to the salt flux is on the order of e-5 (as a FWF), compared to the actual freshwater volume flux change on the order of e-5 (as a FWF). So currently the magnitude of salt fluxes vs freshwater fluxes is approximately equal. BUT this is just for one experiment - I need to compare the sizes of the anomalies.

In [ ]:
# Anomaly (M - C)
print("**Anomaly (M - C)**")
print("tendency =", (tendency_ave_M - tendency_ave_C).values,
      "\nhorizontal advection =", (adv_hor_ave_M - adv_hor_ave_C).values,
      "\nvertical advection =", (adv_vert_ave_M - adv_vert_ave_C).values,
      "\nsurface forcing =", (surface_ave_M - surface_ave_C).values,
      "\ndiffusion 1 =", (diff_1_ave_M - diff_1_ave_C).values,
      "\ndiffusion 2 =", (diff_2_ave_M - diff_2_ave_C).values)

Anomalies are on same order of magnitudes as regular FWF anomalies (e-6).

## Plotting with freshwater flux terms

Values from FreshwaterBudget_pan01.ipynb notebook.

In [ ]:
data = np.load("/g/data/x77/kc5856/analysis/access-panan-comparison/freshwater_budget_values.npz")

# Control (_C) terms
wfo_C       = data["wfo_C"]
lrunoff_C   = data["lrunoff_C"]
net_melt_C  = data["net_melt_C"]
fprec_C     = data["fprec_C"]
precip_C    = data["precip_C"]
evap_C      = data["evap_C"]
budget_C    = data["budget_C"]

# Meltwater (_M) terms
wfo_M       = data["wfo_M"]
lrunoff_M   = data["lrunoff_M"]
net_melt_M  = data["net_melt_M"]
fprec_M     = data["fprec_M"]
precip_M    = data["precip_M"]
evap_M      = data["evap_M"]
budget_M    = data["budget_M"]

In [ ]:
# --- Control (_C) arrays ---
salt_terms_C = [
    tendency_ave_C.values,
    adv_hor_ave_C.values,
    adv_vert_ave_C.values,
    surface_ave_C.values,
    diff_1_ave_C.values,
    diff_2_ave_C.values
]

fresh_terms_C = [
    wfo_C,
    lrunoff_C,
    net_melt_C,
    fprec_C,
    precip_C,
    evap_C,
    budget_C
]

# --- Melt (_M) arrays ---
salt_terms_M = [
    tendency_ave_M.values,
    adv_hor_ave_M.values,
    adv_vert_ave_M.values,
    surface_ave_M.values,
    diff_1_ave_M.values,
    diff_2_ave_M.values
]

fresh_terms_M = [
    wfo_M,
    lrunoff_M,
    net_melt_M,
    fprec_M,
    precip_M,
    evap_M,
    budget_M
]

In [ ]:
# Labels
salt_labels = ["tendency","adv_hor","adv_vert","surface","diff1","diff2"]
fresh_labels = ["wfo","lrunoff","net_melt","fprec","precip","evap","FW budget"]

# Combine labels and values
all_labels = salt_labels + fresh_labels
control_vals = salt_terms_C + fresh_terms_C
melt_vals    = salt_terms_M + fresh_terms_M

In [ ]:
# Plot
x = np.arange(len(all_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14,6))

# Salt terms color: blue, Freshwater: green
colors_control = ["blue"]*len(salt_labels) + ["orange"]*len(fresh_labels)
colors_melt    = ["cornflower"]*len(salt_labels) + ["lightorange"]*len(fresh_labels)

# Plot bars side by side
ax.bar(x - width/2, [v if np.isscalar(v) else v.item() for v in control_vals], width, color=colors_control, label="Control (_C)")
ax.bar(x + width/2, [v if np.isscalar(v) else v.item() for v in melt_vals], width, color=colors_melt, label="Melt (_M)")

# Labels and styling
ax.set_xticks(x)
ax.set_xticklabels(all_labels, rotation=45, ha="right")
ax.set_ylabel("Freshwater flux (kg m-2 s-1)")
ax.set_title("Salt vs Freshwater Terms: Control vs Melt")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()